In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.master('local[*]').appName('test').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/23 20:41:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/23 20:41:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2025-07-23 02:50:15--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-07-22T19%3A49%3A10Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-07-22T18%3A48%3A19Z&ske=2025-07-22T19%3A49%3A10Z&sks=b&skv=2018-11-09&sig=SiaUv7kIT%2Bx5txEhP5Hym449%2F0kRxSZvcdEZUjlyo3Y%3D&jwt=eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1MzIxMDUxNSwibmJmIjoxNzUzMjEwMjE1LCJw

In [5]:
!gunzip ./fhvhv_tripdata_2021-01.csv.gz

In [7]:
!head -n 10 ./fhvhv_tripdata_2021-01.csv > head.csv

In [8]:
!head head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [18]:
from pyspark.sql.types import StringType, TimestampType, IntegerType, StructField, StructType
schema = StructType([
    StructField('hvfhs_license_num', StringType(), True), 
    StructField('dispatching_base_num', StringType(), True), 
    StructField('pickup_datetime', TimestampType(), True), 
    StructField('dropoff_datetime', TimestampType(), True), 
    StructField('PULocationID', IntegerType(), True), 
    StructField('DOLocationID', IntegerType(), True), 
    StructField('SR_Flag', StringType(), True)]
)

df = spark.read\
            .option("header", "true")\
            .schema(schema)\
            .csv("fhvhv_tripdata_2021-01.csv")

In [19]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [26]:
df = df.repartition(24)

In [31]:
df.write.mode('overwrite').parquet('fhvhv/2021/01/')

25/07/23 03:21:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/23 03:21:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/23 03:21:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/23 03:21:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/23 03:21:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [30]:
spark.getActiveSession()

In [33]:
!ls -lh fhvhv/2021/01/

total 436152
-rw-r--r--@ 1 shawn  staff     0B 23 Jul 03:21 _SUCCESS
-rw-r--r--@ 1 shawn  staff   8.9M 23 Jul 03:21 part-00000-5fa7d876-f634-4a49-83bf-738dc1b57ed6-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff   8.9M 23 Jul 03:21 part-00001-5fa7d876-f634-4a49-83bf-738dc1b57ed6-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff   8.9M 23 Jul 03:21 part-00002-5fa7d876-f634-4a49-83bf-738dc1b57ed6-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff   8.9M 23 Jul 03:21 part-00003-5fa7d876-f634-4a49-83bf-738dc1b57ed6-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff   8.9M 23 Jul 03:21 part-00004-5fa7d876-f634-4a49-83bf-738dc1b57ed6-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff   8.9M 23 Jul 03:21 part-00005-5fa7d876-f634-4a49-83bf-738dc1b57ed6-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff   8.9M 23 Jul 03:21 part-00006-5fa7d876-f634-4a49-83bf-738dc1b57ed6-c000.snappy.parquet
-rw-r--r--@ 1 shawn  staff   8.9M 23 Jul 03:21 part-00007-5fa7d876-f634-4a49-83bf-738dc1b57ed6-c000.snappy.parquet
-rw-r--r--@